# 06 · Tổng hợp: Quyết định cho QA, KPI và LLM

Notebook này không chạy code mới. Mỗi dòng dưới đây là một quyết định nhóm cần đưa ra, kèm bằng chứng và vị trí của ô tính trong notebook 01 đến 05
(ví dụ "02 §5" là mục 5 của notebook 02). Các con số là của bộ dữ liệu hiện tại. Với một snapshot mới, hãy chạy lại các notebook 01 đến 05 thay vì
dùng lại bảng này. Phần "đề xuất" là gợi ý của người khảo sát, nhóm cần chốt.

## 1. Quyết định cho QA

| Vấn đề quan sát được | Xem tại | Đề xuất xử lý |
|---|---|---|
| Số bản ghi toàn thành phố ở mốc 2026-03-21 giảm bất thường (53.639 listing, khoảng 19,6 triệu dòng calendar, 1.600.972 review), chưa rõ nguyên nhân. Tập phân công vẫn đủ ID | 01 §2, 02 §1, 03 §1, 04 §1 | Không dùng số liệu toàn thành phố của mốc này làm mẫu số, ghi chú trong báo cáo |
| Mỗi mốc có đủ 1.000 listing không trùng `id`, 365.000 dòng calendar không trùng `(listing_id, date)`, và 5.000 review đúng ID, đúng listing | 02 §1, 03 §1, 04 §1 | Vẫn kiểm lại trong QA khi chạy mốc mới. Config hiện ghi `missing_id_policy: report`, tức ghi danh sách ID thiếu vào báo cáo QA rồi chạy tiếp (chưa có code trong `src/` đọc khóa này). Nếu nhóm muốn dừng pipeline ngay khi thiếu ID, đề xuất đổi sang `error` |
| Số cột tăng từ 79 lên 85 rồi 90, chỉ thêm chứ không bớt | 02 §1 | Đọc cột theo tên, chỉ bắt buộc những cột thực dùng |
| `neighbourhood_group_cleansed`, `license`, `calendar_updated` rỗng ở mọi mốc | 02 §3 | Bỏ qua, dùng `neighbourhood_cleansed` cho quận |
| `neighborhood_overview` rỗng toàn bộ từ 2025-12-13. `host_since`, `host_response_rate`, `host_acceptance_rate`, `instant_bookable` và vài cột chủ nhà khác rỗng toàn bộ từ 2026-03-21. `host_verifications` chỉ còn chuỗi `"None"` | 02 §3 | Gắn cờ cột bị bỏ trống, không so sánh các cột này theo thời gian. `host_is_superhost` vẫn có dữ liệu ở 999/1.000 listing mỗi mốc năm 2026 |
| Điểm review thiếu đúng ở các listing có 0 review (50, 30, 25, 25) | 02 §3 | Để trống, loại khỏi mẫu số của điểm trung bình |
| Giá rỗng ở 53, 1.000, 17, 20 listing. `estimated_revenue_l365d` thiếu đúng ở các dòng thiếu giá | 02 §3, §5 | Chỉ dùng giá và doanh thu ước tính ở mốc cuối |
| Cột `price` ghi ký hiệu `$` nhưng 980 báo giá ở mốc cuối đều ghi GBP và khớp giá mỗi đêm | 02 §4 | Bỏ ký tự, không quy đổi tỷ giá |
| Ngưỡng `price_max_gbp: 5000000` không bắt được dòng nào. Quy tắc IQR trên log(giá) cho ngưỡng khoảng 1.400 GBP, bắt 10 listing | 02 §5 | Nhóm chốt ngưỡng, gắn cờ thay vì xóa, tính KPI giá có và không có các dòng bị cờ |
| `number_of_reviews` giảm ở 2 listing | 02 §8 | Gắn cờ biến cộng dồn bị giảm |
| Tọa độ thay đổi ở 26 listing, xa nhất khoảng 1,3 km. Ở mốc đầu có 2 listing nằm ngoài khung của vùng được giao | 02 §8, 05 §4 | Dùng `analysis_area` cố định, chỉ gắn cờ |
| Độ trễ lớn nhất của `last_scraped` là 4, 20, 12, 12 ngày. Số dòng `previous scrape` là 52, 52, 0, 29 | 02 §9 | Gắn cờ dữ liệu kế thừa, mô tả mốc chụp là một khoảng ngày |
| Có 21, 23, 4, 6 listing mà cả 365 ngày đều không sẵn có | 03 §4 | Gắn cờ `flag_365_ngay_khong_san_co`, trình bày tỷ lệ ngày sẵn có theo cả hai cách tính |
| Khoảng 23% review chứa thẻ `<br/>` | 04 §3 | Thay bằng khoảng trắng ở cột đã làm sạch, giữ cột gốc |

## 2. Quyết định cho KPI

| Vấn đề quan sát được | Xem tại | Đề xuất xử lý |
|---|---|---|
| Mẫu gồm 1.000 listing cố định, 875 nguyên căn và 125 phòng riêng (thành phố khoảng 66% và 33%), có mặt ở 33 quận với 1 đến 168 listing mỗi quận | 01 §5, 02 §6 | Mọi kết luận nói trong phạm vi tập được giao, KPI theo quận ghi kèm `n`, gộp hoặc bỏ quận dưới 10 listing khi xếp hạng |
| Giá mốc cuối lệch phải mạnh, trung vị khoảng 232 GBP, nguyên căn khoảng 258 và phòng riêng khoảng 83 | 02 §5 | Dùng trung vị, thang log, tách theo loại phòng |
| Điểm review dồn sát 5, trung vị khoảng 4,8 | 02 §7 | Điểm số ít phân biệt, nên kết hợp với kết quả LLM |
| `available = f` không phân biệt khách đã đặt với chủ nhà chặn lịch | 03 §3 | Gọi đúng tên "tỷ lệ ngày được ghi là sẵn có", không gọi là tỷ lệ lấp đầy |
| Cùng một tháng cho tỷ lệ khác nhau tùy mốc nhìn. Chỉ tháng 7 và tháng 8/2026 được cả bốn mốc phủ đầy đủ | 03 §3 | So sánh giữa các mốc bằng cùng một khoảng 30 hoặc 90 ngày, khớp với `availability_30` và `availability_90` |
| Chênh lệch giữa các mốc lớn hơn giữa các vùng. A07 có tỷ lệ 30 ngày đầu thấp nhất ở 3 trên 4 mốc | 03 §5 | So sánh vùng trong cùng một mốc |
| Hai cách chia vùng không quy đổi được (Westminster nằm trong 6 vùng A, mỗi vùng A chứa 5 đến 10 quận). Mỗi vùng A có 121 đến 130 listing | 05 §3 | Chọn một cách chia vùng cho mỗi câu hỏi và ghi rõ trong báo cáo |
| Tên quận của cả 1.000 listing đều có trong GeoJSON, và cả 33 quận đều có listing | 05 §2 | Ghép số liệu vào bản đồ theo tên quận, chú thích giới hạn của tọa độ đã làm mờ |

## 3. Quyết định cho LLM và đánh giá

| Vấn đề quan sát được | Xem tại | Đề xuất xử lý |
|---|---|---|
| Bốn mốc có cùng 5.000 review với cùng listing, nội dung và ngày viết. Chỉ tên hiển thị của người viết khác ở 24 review khi so mốc đầu với mốc cuối. Review mới nhất viết ngày 15/9/2025 | 04 §2 | Chạy LLM một lần, xét thay đổi theo thời gian bằng ngày viết review |
| 5.000 review phủ 701 listing, mẫu LLM phủ 375 listing, mẫu nhãn tay phủ 85 listing. Trong mẫu LLM, 38 listing đông review nhất chiếm khoảng 36% số review | 01 §3 | Nói rõ KPI tính trên tập nào, tính theo listing trước khi lấy trung bình |
| Review ngắn nhất dài 64 ký tự, trong khi khoảng 17,5% review của toàn London ngắn hơn mức này | 04 §3 | Ghi rõ mẫu không đại diện cho review rất ngắn |
| Phép dò từ tiếng Anh thông dụng không thấy dấu hiệu rõ của ngôn ngữ khác, nhưng chưa đo được tỷ lệ ngôn ngữ | 04 §4 | Xác định ngôn ngữ bằng LLM, kiểm lại trên nhãn tay, xem riêng review không phải tiếng Anh |
| Ba tập review tương tự nhau về độ dài, năm viết và tỷ lệ HTML, nhưng điều đó chưa chứng minh được chọn ngẫu nhiên hay đại diện về cảm xúc | 04 §5 | Với 100 nhãn tay, chênh lệch vài điểm accuracy cần diễn giải thận trọng, kèm số review đúng và sai |
| Theo bộ từ khóa tự chọn, chỉ vị trí vượt 50% số review (khoảng 63,5%), còn an toàn và giá trị chỉ khoảng 5% | 04 §6 | Ghi số trường hợp của từng khía cạnh trong 100 nhãn tay, thận trọng với F1 của khía cạnh hiếm |
| Phần comments của mẫu LLM khoảng 69 nghìn token, của toàn thành phố khoảng 137 triệu token (quy ước 4 ký tự mỗi token) | 04 §7 | Chạy thử vài lượt để lấy số token thật và cộng thêm prompt, đầu ra khi ước tính chi phí |

## 4. Những gì bộ khảo sát này chưa làm

Bộ notebook là khảo sát dữ liệu, chưa phải QA chính thức. Khi viết QA theo đề, nhóm vẫn cần các quy tắc riêng cho dòng trùng trong một mốc và giữa
các mốc, cho giá trị sai kiểu và cho danh mục lệch (chẳng hạn `room_type` hay `property_type` có giá trị ngoài danh sách mong đợi), cùng báo cáo số
dòng bị ảnh hưởng và KPI trước và sau QA trong `reports/qa_report.csv`. Mỗi quy tắc cần đủ tên, điều kiện, lý do và hành động.

Các hình trong notebook dùng để xem dữ liệu và thử cách trình bày. Hình nộp kèm báo cáo cần được pipeline tự sinh vào `figures/`.

Một số bất thường mới được ghi nhận mà chưa rõ nguyên nhân, gồm mức giảm số bản ghi ở mốc 2026-03-21, các cột bị bỏ trống từ một mốc trở đi,
các mức giá cực cao và việc tọa độ thay đổi giữa các mốc. Báo cáo nên mô tả đúng điều quan sát được và không khẳng định nguyên nhân khi chưa có nguồn.

---
*Bộ notebook này chỉ đọc dữ liệu và không ghi file nào vào `data/`, `reports/` hay `figures/`.*